# 05 — Gerar, revisar e observar um rascunho de política

Vamos criar e configurar os serviços durante as aulas, observar as
respostas da AWS e testar cada componente antes de avançar.

Referência: [workshops AgentCore](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops).

**Pré-requisitos:** uma conta AWS autorizada, região escolhida e o kernel
**Santander Workshop (.venv)** preparado por `scripts/setup.sh`.
Use seu ambiente de notebooks na AWS. Os serviços, ferramentas e agentes
desta jornada executam na AWS.

### Abrir uma sessão AWS e conferir a identidade

O SDK usa a sessão autorizada do seu ambiente AWS. STS informa quem está executando e em qual conta. A região é uma escolha da aula; ajuste-a antes de criar recursos. Nenhuma chave deve ser colada no notebook.

In [ ]:
import json
import os
import uuid
from pathlib import Path

import boto3
from IPython.display import display

REGION = boto3.Session().region_name or "us-west-2"
aws = boto3.Session(region_name=REGION)
sts = aws.client("sts")
identity = sts.get_caller_identity()
ACCOUNT_ID = identity["Account"]
expected_account = os.environ.get("WORKSHOP_EXPECTED_ACCOUNT")
if expected_account and ACCOUNT_ID != expected_account:
    raise RuntimeError(
        "WORKSHOP_ACCOUNT_MISMATCH: a sessão não corresponde à conta esperada."
    )
PARTITION = identity["Arn"].split(":")[1]
print({
    "account": ACCOUNT_ID,
    "region": REGION,
    "principal": identity["Arn"],
    "request_id": identity["ResponseMetadata"]["RequestId"],
})

### Registrar identificadores desta execução

As próximas aulas precisam lembrar os IDs retornados pelas chamadas de criação. Este registro contém apenas identificadores, está vinculado à conta/região e permite reiniciar o kernel. Ele é criado agora: você não precisa receber um arquivo privado do instrutor. O helper abaixo não chama AWS nem cria serviços.

In [ ]:
from santander_demo.notebook_state import NotebookState

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").is_file()
)
state = NotebookState(
    ROOT / ".workshop", account_id=ACCOUNT_ID, region=REGION,
)
NAME_PREFIX = state.prefix
TAGS = state.tags
print({
    "execution": state.data["run_id"],
    "resource_name_prefix": NAME_PREFIX,
    "recorded_resources": sorted(state.data["resources"]),
    "execution_status": state.data.get("status", "ACTIVE"),
})

### Da intenção em linguagem natural a uma política revisável

`Intenção → geração no AgentCore Policy → Cedar e findings → testes → LOG_ONLY`

**Pré-requisito:** aula 04 concluída, com engine ACTIVE, Gateway ENFORCE
e política manual ACTIVE. Esta aula usa **uma conta AWS**. A geração
é feita pelo serviço AgentCore Policy; não executamos um modelo no
notebook. A sessão precisa de `StartPolicyGeneration`, operações de
consulta, `InvokeGateway` para o contexto da ferramenta e `CreatePolicy`.

O serviço gera um rascunho. Revisão de código, validação contra o schema
e testes têm papéis diferentes. A nova política será criada em
**LOG_ONLY**, para observação; a regra manual da aula 04 continua
decidindo as chamadas. A aula não promove automaticamente texto gerado
para ACTIVE.

Os assets gerados expiram após sete dias. Uma política criada a partir
deles permanece até sua exclusão. Usaremos o schema do target real,
criado na aula 01, para dar contexto à geração.

### Conferir as dependências construídas nas aulas anteriores

In [ ]:
import hashlib
import time
import httpx
import jwt

control = aws.client("bedrock-agentcore-control")
iam = aws.client("iam")
cognito = aws.client("cognito-idp")
secrets = aws.client("secretsmanager")
pool_entry = state.require("identity.pool", lesson="01")
client_entry = state.require("identity.client", lesson="01")
gateway_entry = state.require("gateway.instance", lesson="01")
target_entry = state.require("gateway.lambda_target", lesson="01")
role_entry = state.require("gateway.role", lesson="01")
secret_entry = state.require("identity.demo_credentials", lesson="01")
analyst_entry = state.require("identity.user.analista", lesson="01")
auditor_entry = state.require("identity.user.auditor", lesson="01")

def owned(tags):
    assert all(tags.get(key) == value for key, value in TAGS.items()), (
        "POLICY_RESOURCE_OWNERSHIP_MISMATCH"
    )

def wait_ready(read, ready):
    deadline = time.monotonic() + 300
    while time.monotonic() < deadline:
        current = read()
        if current["status"] == ready:
            return current
        if current["status"] in {
            "FAILED", "CREATE_FAILED", "UPDATE_FAILED", "UPDATE_UNSUCCESSFUL",
            "DELETE_FAILED", "DELETING",
        }:
            raise RuntimeError(
                "POLICY_RESOURCE_FAILED: " + str(current.get("statusReasons", []))
            )
        time.sleep(5)
    raise TimeoutError("POLICY_RESOURCE_NOT_READY")

pool = cognito.describe_user_pool(UserPoolId=pool_entry["id"])["UserPool"]
owned(pool["UserPoolTags"])
assert pool["Arn"] == pool_entry["arn"]
POOL_ID = pool_entry["id"]
CLIENT_ID = client_entry["id"]
for entry in (client_entry, analyst_entry, auditor_entry):
    assert entry["pool_id"] == POOL_ID
ISSUER = f"https://cognito-idp.{REGION}.amazonaws.com/{POOL_ID}"
gateway = control.get_gateway(gatewayIdentifier=gateway_entry["id"])
owned(control.list_tags_for_resource(resourceArn=gateway_entry["arn"])["tags"])
assert gateway["gatewayArn"] == gateway_entry["arn"]
assert gateway["name"] == NAME_PREFIX + "-gateway"
assert gateway["status"] == "READY"
assert gateway["roleArn"] == role_entry["arn"]
assert gateway["authorizerType"] == "CUSTOM_JWT"
assert gateway["authorizerConfiguration"] == {"customJWTAuthorizer": {
    "discoveryUrl": ISSUER + "/.well-known/openid-configuration",
    "allowedAudience": [CLIENT_ID],
}}
role = iam.get_role(RoleName=role_entry["name"])["Role"]
owned({tag["Key"]: tag["Value"] for tag in role["Tags"]})
assert role["Arn"] == role_entry["arn"]
assert target_entry["gateway_id"] == gateway_entry["id"]
target = control.get_gateway_target(
    gatewayIdentifier=gateway_entry["id"], targetId=target_entry["id"],
)
assert target["status"] == "READY"
assert target["targetConfiguration"]["mcp"]["lambda"]["lambdaArn"] == (
    state.require("gateway.tool_lambda", lesson="01")["arn"]
)
TOOL_NAME = target_entry["name"] + "___customer_risk"
GATEWAY_URL = gateway["gatewayUrl"]
from santander_demo.aws_endpoints import gateway_region
assert gateway_region(GATEWAY_URL) == REGION
print({"gateway_id": gateway_entry["id"], "tool": TOOL_NAME, "region": REGION})

### Conferir a política que continua protegendo a ferramenta

Antes de gerar algo novo, lemos o engine e a regra manual. O modo ENFORCE do Gateway e o modo ACTIVE da política não serão alterados por esta aula.

In [ ]:
engine_entry = state.require("policy.engine", lesson="04")
active_entry = state.require("policy.customer", lesson="04")
engine = control.get_policy_engine(policyEngineId=engine_entry["id"])
assert engine["policyEngineArn"] == engine_entry["arn"]
assert engine["status"] == "ACTIVE"
owned(control.list_tags_for_resource(resourceArn=engine_entry["arn"])["tags"])
assert gateway["policyEngineConfiguration"] == {
    "arn": engine_entry["arn"], "mode": "ENFORCE",
}
assert active_entry["engine_id"] == engine_entry["id"]
active = control.get_policy(policyEngineId=engine_entry["id"], policyId=active_entry["id"])
assert active["policyArn"] == active_entry["arn"]
assert active["status"] == "ACTIVE" and active["enforcementMode"] == "ACTIVE"
assert hashlib.sha256(
    active["definition"]["cedar"]["statement"].encode(),
).hexdigest() == active_entry["statement_sha256"]
print({"engine_id": engine_entry["id"], "existing_policy": active_entry["id"]})

### Escrever a intenção e iniciar a geração na AWS

O texto usa os IDs efetivamente criados, o nome MCP completo e o parâmetro da ferramenta. Não contém senha, token ou dados de clientes reais. StartPolicyGeneration recebe também o ARN do Gateway, que fornece o contexto do schema.

In [ ]:
INTENT = (
    "Generate a Cedar permit policy for exactly this authorization: "
    f"principal must be AgentCore::OAuthUser::{json.dumps(analyst_entry['sub'])}; "
    f"action must be AgentCore::Action::{json.dumps(TOOL_NAME)}; "
    f"resource must be AgentCore::Gateway::{json.dumps(gateway_entry['arn'])}. "
    "The principal must have OAuth tags iss, aud and token_use. "
    f"principal.getTag(\"iss\") must equal {json.dumps(ISSUER)}, "
    f"principal.getTag(\"aud\") must equal {json.dumps(CLIENT_ID)}, "
    "and principal.getTag(\"token_use\") must equal \"id\". "
    "context.input.customer_id must equal \"cliente-001\". "
    "Do not grant access to other people, tools, gateways, issuers, audiences or customers."
)
print(INTENT)
intent_sha256 = hashlib.sha256(INTENT.encode()).hexdigest()
generation_entry = state.get("policy.generation")
if generation_entry is None:
    generated = control.start_policy_generation(
        policyEngineId=engine_entry["id"],
        name=NAME_PREFIX.replace("-", "_") + "_draft",
        resource={"arn": gateway_entry["arn"]}, content={"rawText": INTENT},
        clientToken=str(uuid.uuid5(uuid.NAMESPACE_URL, NAME_PREFIX + ":policy-generation")),
    )
    generation_entry = state.record("policy.generation", {
        "id": generated["policyGenerationId"], "arn": generated["policyGenerationArn"],
        "engine_id": engine_entry["id"], "gateway_arn": gateway_entry["arn"],
        "intent_sha256": intent_sha256,
    }, lesson="05")
assert generation_entry["engine_id"] == engine_entry["id"]
assert generation_entry["gateway_arn"] == gateway_entry["arn"]
assert generation_entry["intent_sha256"] == intent_sha256, "GENERATION_INTENT_CHANGED"
deadline = time.monotonic() + 300
while True:
    generation = control.get_policy_generation(
        policyEngineId=engine_entry["id"], policyGenerationId=generation_entry["id"],
    )
    assert generation["policyGenerationArn"] == generation_entry["arn"]
    assert generation["resource"] == {"arn": gateway_entry["arn"]}
    if generation["status"] == "GENERATED":
        break
    if generation["status"] != "GENERATING":
        raise RuntimeError("POLICY_GENERATION_FAILED: " + str(generation["statusReasons"]))
    if time.monotonic() >= deadline:
        raise TimeoutError("POLICY_GENERATION_TIMEOUT")
    time.sleep(5)
print({"generation_id": generation_entry["id"], "status": generation["status"]})

### Ler o Cedar gerado e examinar os findings

ListPolicyGenerationAssets retorna o código, a intenção associada e os findings. Exibimos todos os assets. Este exercício espera uma única regra sem findings. A API pode usar definition.cedar ou definition.policy; este último também admite Dogwood. Só prosseguimos se o texto puder ser analisado como Cedar e passar os contraexemplos da próxima célula. Uma extensão incompatível exige revisão; não usamos IGNORE_ALL_FINDINGS.

In [ ]:
assets = []
page = control.list_policy_generation_assets(
    policyEngineId=engine_entry["id"], policyGenerationId=generation_entry["id"],
)
while True:
    assets.extend(page.get("policyGenerationAssets", []))
    if not page.get("nextToken"):
        break
    page = control.list_policy_generation_assets(
        policyEngineId=engine_entry["id"], policyGenerationId=generation_entry["id"],
        nextToken=page["nextToken"],
    )
for asset in assets:
    print(json.dumps({
        "asset_id": asset["policyGenerationAssetId"],
        "intent_fragment": asset["rawTextFragment"],
        "definition": asset.get("definition"), "findings": asset["findings"],
    }, indent=2))
assert len(assets) == 1, "REVIEW_REQUIRED: expected one generated rule"
asset = assets[0]
assert not asset["findings"], "REVIEW_REQUIRED: policy generation findings"
definition = asset.get("definition", {})
assert set(definition) in ({"cedar"}, {"policy"}), (
    "REVIEW_REQUIRED: expected a single policy statement"
)
definition_type = next(iter(definition))
GENERATED_CEDAR = definition[definition_type]["statement"]
assert GENERATED_CEDAR.strip(), "EMPTY_GENERATED_POLICY"
print({"response_definition_type": definition_type, "next_step": "PARSE_AND_TEST_AS_CEDAR"})

### Verificar a intenção com o avaliador Cedar

Estes testes de linguagem não chamam a ferramenta nem substituem o teste AWS. Conferem exemplos permitidos e proibidos, incluindo claims ausentes, outra ferramenta e outro Gateway. Uma matriz finita não prova equivalência para todos os pedidos; por isso o rascunho continuará em observação.

In [ ]:
import cedarpy

parsed = cedarpy.PolicySet.from_str(GENERATED_CEDAR)
cases = [
    ("allowed", {}, True),
    ("other_person", {"person": auditor_entry["sub"]}, False),
    ("other_customer", {"customer": "cliente-002"}, False),
    ("other_issuer", {"iss": "https://untrusted.invalid"}, False),
    ("other_audience", {"aud": "other-app"}, False),
    ("wrong_token_use", {"token_use": "access"}, False),
    ("missing_issuer", {"iss": None}, False),
    ("missing_audience", {"aud": None}, False),
    ("missing_token_use", {"token_use": None}, False),
    ("other_tool", {"tool": "other___tool"}, False),
    ("other_gateway", {"gateway": gateway_entry["arn"] + "-other"}, False),
]
for case, changes, expected in cases:
    person = changes.get("person", analyst_entry["sub"])
    principal = {"type": "AgentCore::OAuthUser", "id": person}
    tags = {"iss": ISSUER, "aud": CLIENT_ID, "token_use": "id"}
    tags.update({key: value for key, value in changes.items() if key in tags})
    request = {
        "principal": principal,
        "action": {"type": "AgentCore::Action", "id": changes.get("tool", TOOL_NAME)},
        "resource": {
            "type": "AgentCore::Gateway",
            "id": changes.get("gateway", gateway_entry["arn"]),
        },
        "context": {"input": {"customer_id": changes.get("customer", "cliente-001")}},
    }
    decision = cedarpy.is_authorized(request, parsed, [{
        "uid": principal, "attrs": {}, "parents": [],
        "tags": {key: value for key, value in tags.items() if value is not None},
    }])
    assert not decision.diagnostics.errors, (
        "GENERATED_POLICY_EVALUATION_ERRORS", case, decision.diagnostics.errors,
    )
    assert decision.allowed == expected, "GENERATED_POLICY_INTENT_MISMATCH: " + case
    print({"cedar_case": case, "allowed": decision.allowed, "execution": "LANGUAGE_VALIDATION"})

### Criar a política gerada em LOG_ONLY

A validação estrita do serviço continua habilitada. LOG_ONLY permite observar esta política sem fazê-la conceder acesso. A política manual permanece ACTIVE e o Gateway ENFORCE. Promover o rascunho exige uma decisão posterior de revisão.

In [ ]:
draft_hash = hashlib.sha256(GENERATED_CEDAR.encode()).hexdigest()
draft_entry = state.get("policy.draft")
if draft_entry is None:
    created = control.create_policy(
        policyEngineId=engine_entry["id"], name="ReviewAnalystCustomer001",
        description="Rascunho gerado pela AWS; apenas observacao, sem aprovacao automatica",
        definition={"cedar": {"statement": GENERATED_CEDAR}},
        validationMode="FAIL_ON_ANY_FINDINGS", enforcementMode="LOG_ONLY",
        clientToken=str(uuid.uuid5(uuid.NAMESPACE_URL, NAME_PREFIX + ":draft-policy")),
    )
    draft_entry = state.record("policy.draft", {
        "id": created["policyId"], "arn": created["policyArn"],
        "engine_id": engine_entry["id"], "statement_sha256": draft_hash,
        "generation_id": generation_entry["id"],
        "asset_id": asset["policyGenerationAssetId"],
    }, lesson="05")
assert draft_entry["engine_id"] == engine_entry["id"]
assert draft_entry["generation_id"] == generation_entry["id"]
assert draft_entry["asset_id"] == asset["policyGenerationAssetId"]
assert draft_entry["statement_sha256"] == draft_hash
draft = wait_ready(
    lambda: control.get_policy(
        policyEngineId=engine_entry["id"], policyId=draft_entry["id"],
    ), "ACTIVE",
)
assert draft["policyArn"] == draft_entry["arn"]
assert draft["definition"] == {"cedar": {"statement": GENERATED_CEDAR}}
assert draft["enforcementMode"] == "LOG_ONLY"
current_gateway = control.get_gateway(gatewayIdentifier=gateway_entry["id"])
assert current_gateway["policyEngineConfiguration"] == {
    "arn": engine_entry["arn"], "mode": "ENFORCE",
}
current_active = control.get_policy(
    policyEngineId=engine_entry["id"], policyId=active_entry["id"],
)
assert current_active["enforcementMode"] == "ACTIVE"
assert current_active["definition"] == active["definition"]
print({
    "draft_policy_id": draft_entry["id"], "enforcement_mode": draft["enforcementMode"],
    "automatically_approved": False,
})

### Repetir as chamadas reais após adicionar o rascunho

Autenticamos novamente as duas pessoas e repetimos o ALLOW e os
dois DENY da aula 04. Isso verifica que as chamadas continuam
protegidas pela política ACTIVE. Esses resultados não comprovam
decisões do rascunho LOG_ONLY; para estudá-las, configure e examine
os logs de avaliação de Policy.

In [ ]:
jwks = jwt.PyJWKClient(ISSUER + "/.well-known/jwks.json")
credentials = json.loads(
    secrets.get_secret_value(SecretId=secret_entry["arn"])["SecretString"],
)
tokens = {}
try:
    for person, expected in (("analista", analyst_entry), ("auditor", auditor_entry)):
        authentication = cognito.initiate_auth(
            ClientId=CLIENT_ID, AuthFlow="USER_PASSWORD_AUTH",
            AuthParameters={"USERNAME": person, "PASSWORD": credentials[person]},
        )["AuthenticationResult"]
        token = authentication["IdToken"]
        claims = jwt.decode(
            token, jwks.get_signing_key_from_jwt(token).key,
            algorithms=["RS256"], issuer=ISSUER, audience=CLIENT_ID,
            options={"require": ["exp", "iat", "iss", "sub", "aud", "token_use"]},
        )
        assert claims["token_use"] == "id" and claims["sub"] == expected["sub"]
        tokens[person] = token
        print({"person": person, "subject_verified": True, "audience": CLIENT_ID})
finally:
    del credentials
del authentication, token

In [ ]:
def mcp_message(response, request_id):
    if response.headers.get("content-type", "").startswith("text/event-stream"):
        messages = [
            json.loads(line[5:].strip()) for line in response.text.splitlines()
            if line.startswith("data:") and line[5:].strip()
        ]
        matches = [message for message in messages if message.get("id") == request_id]
        assert len(matches) == 1, "MCP_RESPONSE_ID_MISMATCH"
        return matches[0]
    message = response.json()
    assert message.get("id") == request_id, "MCP_RESPONSE_ID_MISMATCH"
    return message

cases = [
    ("analyst_allowed_customer", "analista", "cliente-001", "ALLOW"),
    ("analyst_other_customer", "analista", "cliente-002", "DENY"),
    ("auditor_same_customer", "auditor", "cliente-001", "DENY"),
]
for case, person, customer_id, expected in cases:
    with httpx.Client(
        timeout=90, follow_redirects=False,
        headers={
            "Authorization": "Bearer " + tokens[person],
            "Accept": "application/json, text/event-stream",
        },
    ) as http:
        initialize = http.post(GATEWAY_URL, json={
            "jsonrpc": "2.0", "id": 1, "method": "initialize",
            "params": {
                "protocolVersion": "2025-03-26", "capabilities": {},
                "clientInfo": {"name": "santander-policy", "version": "1"},
            },
        })
        initialize.raise_for_status()
        assert "result" in mcp_message(initialize, 1), "MCP_INITIALIZE_FAILED"
        headers = {"MCP-Protocol-Version": "2025-03-26"}
        if initialize.headers.get("mcp-session-id"):
            headers["Mcp-Session-Id"] = initialize.headers["mcp-session-id"]
        response = http.post(GATEWAY_URL, headers=headers, json={
            "jsonrpc": "2.0", "id": 2, "method": "tools/call",
            "params": {"name": TOOL_NAME, "arguments": {"customer_id": customer_id}},
        })
        aws_request_id = response.headers.get("x-amzn-requestid")
        assert aws_request_id, "GATEWAY_RESPONSE_WITHOUT_AWS_REQUEST_ID"
        message = mcp_message(response, 2)
        if expected == "ALLOW":
            response.raise_for_status()
            assert "error" not in message and not message["result"].get("isError")
            result = message["result"]
            data = result.get("structuredContent")
            if data is None:
                text = [item["text"] for item in result["content"] if item["type"] == "text"]
                assert len(text) == 1
                data = json.loads(text[0])
            assert data["execution"] == "AWS_LAMBDA" and data["synthetic"] is True
            assert data["customer_id"] == customer_id and data["risk"] == "low"
            assert data["lambda_request_id"]
            print({
                "case": case, "status": "ALLOW", "request_id": aws_request_id,
                "lambda_request_id": data["lambda_request_id"],
            })
        else:
            # A resposta precisa identificar a decisão de Policy.
            # Outros erros não são convertidos em um teste aprovado.
            error = message.get("error", {})
            result = message.get("result", {})
            assert bool(error) != bool(result.get("isError")), "EXPECTED_MCP_ERROR"
            if error:
                assert "result" not in message
                denial_message = error.get("message", "").lower()
            else:
                assert result["isError"] is True
                denial_message = " ".join(
                    item["text"] for item in result.get("content", [])
                    if item.get("type") == "text"
                ).lower()
            assert response.status_code in {200, 403}, (
                "UNEXPECTED_DENIAL_HTTP_STATUS", response.status_code,
            )
            assert "policy" in denial_message and any(
                word in denial_message for word in ("denied", "not allowed", "unauthorized", "deny")
            ), "EXPECTED_POLICY_DENIAL: " + json.dumps(error)
            print({
                "case": case, "status": "DENY", "http_status": response.status_code,
                "mcp_error_code": error.get("code"), "request_id": aws_request_id,
            })
del tokens

### Revisão antes de promover uma política

Compare intenção e Cedar: principal, action, resource, claims e
parâmetros. Examine findings e casos ausentes na matriz. Uma
política gerada pode estar sintaticamente correta e ainda conceder
mais acesso do que o necessário.

Nesta aula criamos o rascunho na AWS em LOG_ONLY e mantivemos a
proteção anterior. O aceite completo exige geração concluída,
assets revisáveis, testes da linguagem e chamadas reais protegidas.
Não há promoção automática a ACTIVE.

Entra ID usa o mesmo processo de autoria e revisão, com issuer,
audiência e claims do seu tenant/aplicação. Não substitua apenas
o nome do provedor: revise o contrato de identidade da política.

A aula 99 remove também a política em observação. Assets de geração
são temporários no serviço e expiram após sete dias; não existe
uma chamada DeletePolicyGeneration no SDK usado nesta jornada.